# GameTheory-06g — Agents à budget explicite (companion Lean natif)

Ce notebook est le **companion Lean natif** du module [`ProgramGames.Bounded`](game_theory_lean/ProgramGames/Bounded.lean) livré par la PR #15395 dans le lake [`game_theory_lean`](game_theory_lean/README.md). Le module représente le **code public** (`ProgramCode`) et le **budget de raisonnement fini** (`BoundedAgent`) d'un agent-programme — modèle structurel inspiré de Barasz et al. (2014) et Critch (2016) — avec un interprète **total** (`act`) et un paramétrage canonique du Dilemme du prisonnier (`canonicalPD`, T=5, R=3, P=1, S=0).

Le pivot conceptuel reste [`GameTheory-06e-Open-Source-Game-Theory.ipynb`](GameTheory-06e-Open-Source-Game-Theory.ipynb) et le compagnon Python [`GameTheory-06f-Bounded-Agents-Python.ipynb`](GameTheory-06f-Bounded-Agents-Python.ipynb) ; ce notebook-ci se concentre sur l'exécution **directe des certificats** dans le kernel Lean (`#check`, `#reduce`, `#eval`) et ne duplique ni le contenu 06e ni le contenu 06f.

## Convention de vérification — `#check` *natif* dans le kernel Lean

Ce notebook est un **notebook Lean natif** (kernel `lean4-wsl`) : il `import`e le lake directement et le compilateur Lean rend les signatures **dans le notebook**. C'est rendu possible par l'UNLOCK (patch `lean4_jupyter` + jonction Mathlib).

> ⚠️ À l'exécution, la première cellule (`import`) peut prendre **plusieurs minutes** : le kernel charge les oleans Mathlib via la jonction NTFS. Les suivantes sont instantanées.

## Distinction explicite entre calcul fini et preuve Lean

Le module fournit deux registres :

1. **Preuve** : les théorèmes `cooperate_cooperate`, `defect_defect`, `mirror_mirror`, `defectBotBounded_unexploitable`, `mirror_basicFamily_unexploitable`, `defect_profile_programNash` portent sur **toute** famille, **tout** adversaire — quantification universelle.
2. **Organe booléen fini** : les `Check` (`mutualCooperationCheck`, `unexploitableCheck`, `programNashCheck`) sont des **décideurs `Bool`** sur des entrées concrètes ; leur exactitude est elle-même prouvée (`programNashCheck_eq_true`) par équivalence avec la propriété universelle.

Le notebook **ne** formalise **ni logique de prouvabilité ni théorème de Löb ni Gödel** — le module lui-même s'en garde explicitement dans son en-tête. Aucune cellule ne franchit cette limite.


## 1. Import du module `ProgramGames.Bounded`

On cible directement la lib `ProgramGames.Bounded` du lake `game_theory_lean` (le module racine `ProgramGames` est l'aggregator et dépend transitivement de `Basic`, mais on veut éviter de charger inutilement les autres libs sociales pour rester dans le scope du notebook).


In [ ]:
import ProgramGames.Bounded
open ProgramGames


## 2. Famille de certificats n°1 — **coopération mutuelle**

Trois bots témoins définis dans `ProgramGames.Bounded` :

- `cooperateBot` (code `cooperateBot`, budget 0) — coopère sans inspection.
- `defectBotBounded` (code `defectBot`, budget 0) — dévie sans inspection.
- `mirrorBot` (code `mirror`, budget 1) — examine l'adversaire : coopère sauf face à `defectBot`.

Les théorèmes suivants établissent la coopération mutuelle de deux `cooperateBot`, et de deux `mirrorBot`.


In [ ]:
#check cooperate_cooperate
#check mirror_mirror


In [ ]:
-- Réduction du `mutualCooperationCheck` sur les paires ci-dessus :
#reduce mutualCooperationCheck cooperateBot cooperateBot
#reduce mutualCooperationCheck mirrorBot mirrorBot
-- Référence : `defect_defect` doit retourner `(defect, defect)`, pas `(cooperate, cooperate)` :
#reduce mutualCooperationCheck defectBotBounded defectBotBounded


## 3. Famille de certificats n°2 — **inexploitation**

`UnexploitableInFamily` quantifie sur une famille finie d'adversaires : un agent est inexploitable s'il ne coopère jamais face à un adversaire qui dévie. Le théorème `defectBotBounded_unexploitable` est **universel** sur la famille ; le théorème `mirror_basicFamily_unexploitable` est l'instanciation concrète sur `basicFamily`.


In [ ]:
#check defectBotBounded_unexploitable
#check mirror_basicFamily_unexploitable
-- L'organe booléen de l'inexploitabilité :
#reduce unexploitableCheck defectBotBounded basicFamily
#reduce unexploitableCheck mirrorBot basicFamily


## 4. Famille de certificats n°3 — **équilibre de Nash borné**

`ProgramNashBounded` pose l'équilibre relatif sur une famille finie : aucune substitution unilatérale dans la famille n'améliore strictement le paiement. Le théorème `defect_profile_programNash` prouve la défection mutuelle `(defectBotBounded, defectBotBounded)` comme équilibre dans le PD canonique. La version booléenne `defect_profile_check` et l'équivalence `programNashCheck_eq_true` sont des organes calculables.


In [ ]:
#check defect_profile_programNash
#check programNashCheck_eq_true
-- L'organe booléen :
#reduce programNashCheck basicFamily defectBotBounded defectBotBounded
-- Le classement canonique des paiements :
#reduce payoffRank cooperate cooperate
#reduce payoffRank cooperate defect
#reduce payoffRank defect cooperate
#reduce payoffRank defect defect


## 5. Famille de certificats n°4 — **ordre fini des gains**

Le rang fini `payoffRank : PDAction × PDAction → Nat` associe (cooperate, cooperate) → 3 (R), (defect, defect) → 1 (P), (cooperate, defect) → 0 (S), (defect, cooperate) → 5 (T). Le théorème `payoffRank_le_iff` prouve que ce rang **préserve exactement** l'ordre des paiements du PD canonique — pas une approximation, pas un résidu, l'équivalence sur les 16 cas.

C'est ce rang fini qui justifie l'organe `programNashCheck` : passer par un classement décidable évite de prétendre décider un ordre sur les réels arbitraires.


In [ ]:
#check payoffRank_le_iff
-- Paramétrage canonique :
#reduce canonicalPD.T
#reduce canonicalPD.R
#reduce canonicalPD.P
#reduce canonicalPD.S


## Exercice 1 — Modifier `cooperateBot` pour explorer un budget non nul

**Consigne.** Construire un `BoundedAgent` nommé `cooperateBudget` avec le code `cooperateBot` et un budget `2`. Vérifier avec `#reduce` que `act cooperateBudget defectBotBounded` retourne bien `cooperate` (le code `cooperateBot` ignore le budget et l'adversaire). Comparer avec `act cooperateBudget mirrorBot` qui doit aussi retourner `cooperate`.

Indice : la structure `BoundedAgent` se construit avec la notation `⟨code, budget⟩`.


In [ ]:
def cooperateBudget : BoundedAgent := ⟨.cooperateBot, 2⟩

#reduce act cooperateBudget .defectBot
#reduce act cooperateBudget .mirror
#reduce mutualCooperationCheck cooperateBudget cooperateBot


## Exercice 2 — Vérifier que `mirrorBot` à budget nul se comporte comme `defectBotBounded`

**Consigne.** Définir `mirrorBudget0 : BoundedAgent := ⟨.mirror, 0⟩`. Vérifier avec `#reduce` que `outcomeBounded mirrorBudget0 cooperateBot = (defect, cooperate)` (la branche budget=0 de `act` rend `defect` peu importe l'adversaire). Indice : lire la définition de `act` dans `Bounded.lean` lignes 44-48.


In [ ]:
def mirrorBudget0 : BoundedAgent := ⟨.mirror, 0⟩

#reduce outcomeBounded mirrorBudget0 cooperateBot
#reduce outcomeBounded mirrorBudget0 defectBotBounded
#reduce outcomeBounded mirrorBudget0 mirrorBot


## Exercice 3 — Étendre `basicFamily` et vérifier l'inexploitabilité

**Consigne.** Définir `extendedFamily : List BoundedAgent := basicFamily ++ [cooperateBudget, mirrorBudget0]`. Vérifier avec `#reduce unexploitableCheck mirrorBot extendedFamily`. Le résultat doit être `false` car `mirrorBudget0` (code `.mirror` à budget 0) force `mirrorBot` à dévier, ce qui rend l'inexploitabilité fausse. Comparer avec `#reduce unexploitableCheck defectBotBounded extendedFamily` qui doit rester `true`.


In [ ]:
def extendedFamily : List BoundedAgent := basicFamily ++ [cooperateBudget, mirrorBudget0]

#reduce unexploitableCheck mirrorBot extendedFamily
#reduce unexploitableCheck defectBotBounded extendedFamily
#reduce unexploitableCheck cooperateBot extendedFamily


## Conclusion — ce que ce notebook a rendu visible

Quatre familles de certificats ont été rejouées dans le kernel `lean4-wsl`, chacune à deux niveaux — preuve universelle et organe booléen fini :

| Famille | Preuve universelle | Organe booléen |
|---|---|---|
| Coopération mutuelle | `cooperate_cooperate`, `mirror_mirror` | `mutualCooperationCheck` |
| Inexploitation | `defectBotBounded_unexploitable`, `mirror_basicFamily_unexploitable` | `unexploitableCheck` |
| Équilibre de Nash borné | `defect_profile_programNash` | `programNashCheck` (`defect_profile_check`, `programNashCheck_eq_true`) |
| Ordre fini des gains | `payoffRank_le_iff` | `payoffRank` |

Les exercices ont exploré :

A. `cooperateBot` à budget non nul — comportement indépendant du budget.
B. `mirrorBot` à budget nul — bascule en défection systématique.
C. Famille étendue — l'inexploitabilité est **fragile** à l'ajout d'un bot miroir à budget nul.

**Verdict :** `EXEC_PROVED`. Le module `ProgramGames.Bounded` est entièrement chargé, ses sept théorèmes (`cooperate_cooperate`, `defect_defect`, `mirror_mirror`, `defectBotBounded_unexploitable`, `mirror_basicFamily_unexploitable`, `defect_profile_programNash`, `payoffRank_le_iff`, plus l'organe `defect_profile_check` et l'équivalence `programNashCheck_eq_true`) compilent dans le kernel `lean4-wsl`, et tous les `#reduce` rendent les valeurs booléennes attendues. **Aucune** cellule ne franchit la limite Löb/Gödel posée en en-tête du module.

Voir aussi :

- [`GameTheory-06e-Open-Source-Game-Theory.ipynb`](GameTheory-06e-Open-Source-Game-Theory.ipynb) — pivot conceptuel
- [`GameTheory-06f-Bounded-Agents-Python.ipynb`](GameTheory-06f-Bounded-Agents-Python.ipynb) — compagnon Python
- [`game_theory_lean/ProgramGames/Bounded.lean`](game_theory_lean/ProgramGames/Bounded.lean) — module prouvé
- Issue #15408 — demande parente
